# Compute Correlations with HK1 Protein Abundance: REDS Recall
## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from itertools import product

import numpy as np
import pandas as pd
import scipy as sp

from hk1_r12ter_analysis.config import (
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROTEIN_ID,
    logger,
)
from hk1_r12ter_analysis.io import (
    load_dataframe_from_file,
    make_filename,
    save_dataframe_to_file,
)

# Suppress logging if desired
logger.remove(1)
logger.add(sys.stderr, level="WARNING");

## Load REDS Recall data

In [ ]:
sample_key = "RECALL DONOR ID"
group_key = "Day"
source_type = "REDS-Recall"
main_data_type = ("Proteomics", PROTEIN_ID)
other_data_type = "All_Data"
norm_str = "median-none-auto"

# Input data arguments
input_dirpath = PROCESSED_DATA_DIR / "REDS" / "Normalized" / norm_str
# Output data arguments
output_dirpath = PROCESSED_DATA_DIR / "REDS" / "Correlations"
output_dirpath.mkdir(parents=True, exist_ok=True)

### All data

In [ ]:
input_filename = make_filename(source_type, "Donor", "RBC", "All_Data")
df_all_data = load_dataframe_from_file(
    input_dirpath / input_filename,
    index_col=[key for key in [sample_key, group_key] if key],
    header=[0, 1],
)
df_all_data = df_all_data.convert_dtypes()
if group_key:
    # Swap group and samples in index
    df_all_data.index = df_all_data.index.swaplevel(0, 1)
    df_all_data.index = df_all_data.index.set_levels(
        df_all_data.index.levels[0].astype(str), level=0
    )
    groups = df_all_data.index.get_level_values(0).unique().tolist()

df_all_data = df_all_data.select_dtypes(include=np.number)
df_data_main = df_all_data.loc[
    :,
    df_all_data.columns.isin(
        **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
    ),
].copy()
if other_data_type == "All_Data":
    df_data_other = df_all_data.loc[
        :,
        ~df_all_data.columns.isin(
            **dict(values=[main_data_type], level=0 if isinstance(main_data_type, str) else None)
        ),
    ].copy()
else:
    df_data_other = df_all_data.loc[
        :,
        df_all_data.columns.isin(
            **dict(values=[other_data_type], level=0 if isinstance(other_data_type, str) else None)
        ),
    ].copy()
df_data_main

#### Compute correlations with HK1 protein abundance

In [ ]:
# Save to interim directory prevents the need to recompute
recompute = False
nan_policy = "omit"
correlation_statistic = "Spearman"
output_filename = make_filename(
    source_type,
    "_".join(
        [
            main_data_type if isinstance(main_data_type, str) else "-".join(main_data_type),
            other_data_type if isinstance(other_data_type, str) else "-".join(other_data_type),
        ]
    ),
    correlation_statistic,
)

index_cols = ["Group", "DataType1", "Variable1", "DataType2", "Variable2"]
if not recompute:
    logger.debug("Loading previously computed correlations...")
    df_correlations = load_dataframe_from_file(output_dirpath / output_filename, index_col=None)
    df_correlations["Group"] = df_correlations["Group"].astype(str)
    logger.info("Loaded previously computed correlations.")
else:
    logger.info("Computing Spearman correlations...")
    correlations_dict = defaultdict(dict)

    for idx, (group, (col_names_main, series_main), (col_names_other, series_other)) in enumerate(
        product(
            ["ALL"] + (groups if group_key else []), df_data_main.items(), df_data_other.items()
        )
    ):
        if col_names_main == col_names_other:
            continue

        if group != "ALL":
            series_main = series_main.loc[group]
            series_other = series_other.loc[group]
        if len(series_other.unique()) == 1:
            continue
        result = sp.stats.spearmanr(
            series_main.values,
            series_other.values,
            axis=0,
            nan_policy=nan_policy,
            alternative="two-sided",
        )
        correlations_dict[idx]["Group"] = group
        correlations_dict[idx].update(dict(zip(["DataType1", "Variable1"], col_names_main)))
        correlations_dict[idx].update(dict(zip(["DataType2", "Variable2"], col_names_other)))
        correlations_dict[idx][f"{correlation_statistic} statistic"] = result.statistic
        correlations_dict[idx][f"{correlation_statistic} p-value"] = result.pvalue
        correlations_dict[idx][f"{correlation_statistic} -log10(p-value)"] = -np.log10(
            result.pvalue
        )
    df_correlations = pd.DataFrame.from_dict(correlations_dict, orient="index")
    logger.info("Computed Spearman correlations.")

    # Set groups as categories
    if df_correlations["Group"].size > 1:
        df_correlations["Group"] = pd.Categorical(df_correlations["Group"])
        df_correlations["Group"] = df_correlations["Group"].cat.reorder_categories(
            ["ALL"] + (groups if group_key else []), ordered=True
        )
    df_correlations = df_correlations.sort_values(
        ["Group", "DataType1", "DataType2", f"{correlation_statistic} -log10(p-value)"],
        ascending=[True, True, True, False],
    )
    df_correlations = df_correlations.reset_index(drop=True)
    save_dataframe_to_file(df_correlations, output_dirpath / output_filename, index=False)

df_correlations = df_correlations.set_index(index_cols)
df_correlations